In [ ]:
!pip install openmeteo-requests --quiet
!pip install requests-cache retry-requests --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.6/208.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.7/722.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.4/399.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import time

In [ ]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Download Weather Data

## Prepared 58 California counties centroids

In [ ]:
# https://data.ca.gov/dataset/ca-geographic-boundaries
counties_gdf = gpd.read_file("CA_Counties.shp")
counties_gdf.head()

,STATEFP,COUNTYFP,COUNTYNS,GEOID,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,Shape_Leng,Shape_Area,geometry
0,06,091,00277310,06091,Sierra,Sierra County,06,H1,G4020,None,None,None,A,2.468695e+09,2.329911e+07,+39.5769252,-120.5219926,375602.758281,4.200450e+09,"POLYGON ((-13431319.751 4821511.426, -13431312..."
1,06,067,00277298,06067,Sacramento,Sacramento County,06,H1,G4020,472,40900,None,A,2.499984e+09,7.542543e+07,+38.4500161,-121.3404408,406584.174167,4.205516e+09,"POLYGON ((-13490651.476 4680831.603, -13490511..."
2,06,083,00277306,06083,Santa Barbara,Santa Barbara County,06,H1,G4020,None,42200,None,A,7.084063e+09,2.729752e+09,+34.5370572,-120.0399729,891686.747247,1.449841e+10,"MULTIPOLYGON (((-13440081.316 4150394.004, -13..."
3,06,009,01675885,06009,Calaveras,Calaveras County,06,H1,G4020,None,None,None,A,2.641785e+09,4.384187e+07,+38.1838996,-120.5614415,367005.879680,4.356213e+09,"POLYGON ((-13428575.483 4627725.227, -13428534..."
4,06,111,00277320,06111,Ventura,Ventura County,06,H1,G4020,348,37100,None,A,4.771988e+09,9.473454e+08,+34.3587415,-119.1331432,527772.242190,8.413293e+09,"MULTIPOLYGON (((-13283668.94 4059436.934, -132..."


In [ ]:
len(counties_gdf)

58

In [ ]:
# Compute centroids
counties_gdf_geo = counties_gdf.to_crs(epsg=4326)
counties_gdf_geo["centroid"] = counties_gdf_geo.geometry.centroid

# Build lists for API call
county_centroids = []
for _, row in counties_gdf_geo.iterrows():
    county_name = row["NAME"]
    lat = row["centroid"].y
    lon = row["centroid"].x
    county_centroids.append({
        "county": county_name,
        "latitude": lat,
        "longitude": lon
    })

latitudes = [c["latitude"] for c in county_centroids]
longitudes = [c["longitude"] for c in county_centroids]
county_names = [c["county"] for c in county_centroids]

/tmp/ipykernel_61623/3435294628.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  counties_gdf_geo["centroid"] = counties_gdf_geo.geometry.centroid


## Fetch daily weather data from Open-Meteo Historical API

In [ ]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [ ]:
url = "https://archive-api.open-meteo.com/v1/archive"
all_dfs = [] # Initialize a list to store DataFrames for each county

In [ ]:
for i, c in enumerate(county_centroids, 1):
    print(f"[{i:2d}/{len(county_centroids)}] Fetching data for {c['county']} ...")

    # Construct params for a single county
    params = {
        "latitude": c["latitude"],
        "longitude": c["longitude"],
        "start_date": "2010-01-01",
        "end_date": "2020-12-31",
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "windspeed_10m_max",
            "precipitation_sum",
            "dewpoint_2m_mean"
        ],
        "timezone": "America/Los_Angeles"
    }

    try:
        # Use openmeteo_requests.Client for the API call
        responses = openmeteo.weather_api(url, params=params)
        daily = responses[0].Daily()

        # Build the daily date range
        date_range = pd.date_range(
            start=pd.to_datetime(daily.Time(), unit="s"),
            end=pd.to_datetime(daily.TimeEnd(), unit="s"),
            freq=pd.Timedelta(seconds=daily.Interval()),
            inclusive="left"
        )

        # Extract variables by index
        df_county = pd.DataFrame({
            "date": date_range,
            "temp_max": daily.Variables(0).ValuesAsNumpy(),
            "temp_min": daily.Variables(1).ValuesAsNumpy(),
            "wind_speed": daily.Variables(2).ValuesAsNumpy(),
            "precipitation": daily.Variables(3).ValuesAsNumpy(),
        })

        # Humidity from dewpoint_2m_mean
        Td = daily.Variables(4).ValuesAsNumpy()
        T_mean = (df_county["temp_max"] + df_county["temp_min"]) / 2.0
        df_county["humidity"] = (
            100
            * np.exp(17.27 * Td / (Td + 237.3))
            / np.exp(17.27 * T_mean / (T_mean + 237.3))
        )

        df_county["county"] = c["county"]
        all_dfs.append(df_county)

        print(f"    ✓ {c['county']} done ({len(df_county)} days)")

    except Exception as e:
        print(f"    ERROR fetching data for {c['county']}: {e}. Skipping.")

    time.sleep(20) # Avoid spamming API rate limits

[ 1/58] Fetching data for Sierra ...
    ✓ Sierra done (4018 days)
[ 2/58] Fetching data for Sacramento ...
    ✓ Sacramento done (4018 days)
[ 3/58] Fetching data for Santa Barbara ...
    ✓ Santa Barbara done (4018 days)
[ 4/58] Fetching data for Calaveras ...
    ✓ Calaveras done (4018 days)
[ 5/58] Fetching data for Ventura ...
    ✓ Ventura done (4018 days)
[ 6/58] Fetching data for Los Angeles ...
    ✓ Los Angeles done (4018 days)
[ 7/58] Fetching data for Sonoma ...
    ✓ Sonoma done (4018 days)
[ 8/58] Fetching data for Kings ...
    ✓ Kings done (4018 days)
[ 9/58] Fetching data for San Diego ...
    ✓ San Diego done (4018 days)
[10/58] Fetching data for Placer ...
    ✓ Placer done (4018 days)
[11/58] Fetching data for San Francisco ...
    ✓ San Francisco done (4018 days)
[12/58] Fetching data for Marin ...
    ERROR fetching data for Marin: failed to request 'https://archive-api.open-meteo.com/v1/archive': {'error': True, 'reason': 'Minutely API request limit exceeded. Ple

In [ ]:
len(all_dfs)

35

In [ ]:
missing_counties = [
    "Marin", "Solano", "Yuba", "Santa Clara", "Tehama", "San Joaquin",
    "Alameda", "Nevada", "Butte", "Merced", "Tulare", "Stanislaus",
    "Orange", "Imperial", "Sutter", "Amador", "Lake", "Plumas",
    "San Mateo", "Siskiyou", "Santa Cruz", "Glenn", "San Luis Obispo"
]

missing_county_centroids = [c for c in county_centroids if c["county"] in missing_counties]

for i, c in enumerate(missing_county_centroids, 1):
    print(f"[{i:2d}/{len(missing_county_centroids)}] Fetching data for {c['county']} ...")

    # Construct params for a single county
    params = {
        "latitude": c["latitude"],
        "longitude": c["longitude"],
        "start_date": "2010-01-01",
        "end_date": "2020-12-31",
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "windspeed_10m_max",
            "precipitation_sum",
            "dewpoint_2m_mean"
        ],
        "timezone": "America/Los_Angeles"
    }

    try:
        # Use openmeteo_requests.Client for the API call
        responses = openmeteo.weather_api(url, params=params)
        daily = responses[0].Daily()

        # Build the daily date range
        date_range = pd.date_range(
            start=pd.to_datetime(daily.Time(), unit="s"),
            end=pd.to_datetime(daily.TimeEnd(), unit="s"),
            freq=pd.Timedelta(seconds=daily.Interval()),
            inclusive="left"
        )

        # Extract variables by index
        df_county = pd.DataFrame({
            "date": date_range,
            "temp_max": daily.Variables(0).ValuesAsNumpy(),
            "temp_min": daily.Variables(1).ValuesAsNumpy(),
            "wind_speed": daily.Variables(2).ValuesAsNumpy(),
            "precipitation": daily.Variables(3).ValuesAsNumpy(),
        })

        # Humidity from dewpoint_2m_mean
        Td = daily.Variables(4).ValuesAsNumpy()
        T_mean = (df_county["temp_max"] + df_county["temp_min"]) / 2.0
        df_county["humidity"] = (
            100
            * np.exp(17.27 * Td / (Td + 237.3))
            / np.exp(17.27 * T_mean / (T_mean + 237.3))
        )

        df_county["county"] = c["county"]
        all_dfs.append(df_county)

        print(f"    ✓ {c['county']} done ({len(df_county)} days)")

    except Exception as e:
        print(f"    ERROR fetching data for {c['county']}: {e}. Skipping.")

    time.sleep(10) # Avoid spamming API rate limits

[ 1/58] Fetching data for Marin ...
    ✓ Marin done (4018 days)
[ 2/58] Fetching data for Solano ...
    ✓ Solano done (4018 days)
[ 3/58] Fetching data for Yuba ...
    ✓ Yuba done (4018 days)
[ 4/58] Fetching data for Santa Clara ...
    ✓ Santa Clara done (4018 days)
[ 5/58] Fetching data for Tehama ...
    ✓ Tehama done (4018 days)
[ 6/58] Fetching data for San Joaquin ...
    ✓ San Joaquin done (4018 days)
[ 7/58] Fetching data for Alameda ...
    ✓ Alameda done (4018 days)
[ 8/58] Fetching data for Nevada ...
    ✓ Nevada done (4018 days)
[ 9/58] Fetching data for Butte ...
    ✓ Butte done (4018 days)
[10/58] Fetching data for Merced ...
    ✓ Merced done (4018 days)
[11/58] Fetching data for Tulare ...
    ✓ Tulare done (4018 days)
[12/58] Fetching data for Stanislaus ...
    ✓ Stanislaus done (4018 days)
[13/58] Fetching data for Orange ...
    ✓ Orange done (4018 days)
[14/58] Fetching data for Imperial ...
    ERROR fetching data for Imperial: failed to request 'https://arc

In [ ]:
missing_counties_2 = [
    "Imperial", "Siskiyou"
]

missing_county_centroids_2 = [c for c in county_centroids if c["county"] in missing_counties_2]

for i, c in enumerate(missing_county_centroids_2, 1):
    print(f"[{i:2d}/{len(missing_county_centroids_2)}] Fetching data for {c['county']} ...")

    # Construct params for a single county
    params = {
        "latitude": c["latitude"],
        "longitude": c["longitude"],
        "start_date": "2010-01-01",
        "end_date": "2020-12-31",
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "windspeed_10m_max",
            "precipitation_sum",
            "dewpoint_2m_mean"
        ],
        "timezone": "America/Los_Angeles"
    }

    try:
        # Use openmeteo_requests.Client for the API call
        responses = openmeteo.weather_api(url, params=params)
        daily = responses[0].Daily()

        # Build the daily date range
        date_range = pd.date_range(
            start=pd.to_datetime(daily.Time(), unit="s"),
            end=pd.to_datetime(daily.TimeEnd(), unit="s"),
            freq=pd.Timedelta(seconds=daily.Interval()),
            inclusive="left"
        )

        # Extract variables by index
        df_county = pd.DataFrame({
            "date": date_range,
            "temp_max": daily.Variables(0).ValuesAsNumpy(),
            "temp_min": daily.Variables(1).ValuesAsNumpy(),
            "wind_speed": daily.Variables(2).ValuesAsNumpy(),
            "precipitation": daily.Variables(3).ValuesAsNumpy(),
        })

        # Humidity from dewpoint_2m_mean
        Td = daily.Variables(4).ValuesAsNumpy()
        T_mean = (df_county["temp_max"] + df_county["temp_min"]) / 2.0
        df_county["humidity"] = (
            100
            * np.exp(17.27 * Td / (Td + 237.3))
            / np.exp(17.27 * T_mean / (T_mean + 237.3))
        )

        df_county["county"] = c["county"]
        all_dfs.append(df_county)

        print(f"    ✓ {c['county']} done ({len(df_county)} days)")

    except Exception as e:
        print(f"    ERROR fetching data for {c['county']}: {e}. Skipping.")

    time.sleep(10) # Avoid spamming API rate limits

[ 1/2] Fetching data for Imperial ...
    ✓ Imperial done (4018 days)
[ 2/2] Fetching data for Siskiyou ...
    ✓ Siskiyou done (4018 days)


In [ ]:
len(all_dfs)

58

In [ ]:
# Combine everything
weather_df = pd.concat(all_dfs, ignore_index=True)

# Save
weather_df = weather_df[["county", "date", "temp_max", "temp_min", "humidity", "wind_speed", "precipitation"]]
weather_df.to_csv("weather_by_county.csv", index=False)
print(f"Saved {len(weather_df):,} rows to weather_by_county.csv")

Saved 233,044 rows to weather_by_county.csv


# Merge Weather with Base Dataset

In [ ]:
weather_df = pd.read_csv("weather_by_county.csv")
weather_df.head()

,county,date,temp_max,temp_min,humidity,wind_speed,precipitation
0,Sierra,2010-01-01 07:00:00,-0.6035,-6.1035,88.510060,13.044723,3.8
1,Sierra,2010-01-02 07:00:00,3.2465,-6.2535,72.317310,9.199390,0.0
2,Sierra,2010-01-03 07:00:00,3.5965,-9.3535,59.366627,8.557102,0.0
3,Sierra,2010-01-04 07:00:00,3.4465,-7.2035,51.487072,8.145870,0.0
4,Sierra,2010-01-05 07:00:00,4.4965,-5.1535,49.372833,5.241679,0.0


In [ ]:
base_df = pd.read_csv("base_dataset.csv")
base_df['county'] = base_df['county'].str.replace(' County', '')
base_df.head()

,county,date,fire_label,max_frp,max_brightness,fire_count
0,Alameda,2016-01-01,0,0.0,0.0,0
1,Alameda,2016-01-02,0,0.0,0.0,0
2,Alameda,2016-01-03,0,0.0,0.0,0
3,Alameda,2016-01-04,0,0.0,0.0,0
4,Alameda,2016-01-05,0,0.0,0.0,0


In [ ]:
weather_df['date'] = pd.to_datetime(weather_df['date']).dt.date
base_df['date'] = pd.to_datetime(base_df['date']).dt.date

df = pd.merge(base_df, weather_df, on=['county', 'date'], how='right')
df.head()

,county,date,fire_label,max_frp,max_brightness,fire_count,temp_max,temp_min,humidity,wind_speed,precipitation
0,Sierra,2010-01-01,NaN,NaN,NaN,NaN,-0.6035,-6.1035,88.510060,13.044723,3.8
1,Sierra,2010-01-02,NaN,NaN,NaN,NaN,3.2465,-6.2535,72.317310,9.199390,0.0
2,Sierra,2010-01-03,NaN,NaN,NaN,NaN,3.5965,-9.3535,59.366627,8.557102,0.0
3,Sierra,2010-01-04,NaN,NaN,NaN,NaN,3.4465,-7.2035,51.487072,8.145870,0.0
4,Sierra,2010-01-05,NaN,NaN,NaN,NaN,4.4965,-5.1535,49.372833,5.241679,0.0


In [ ]:
# Check missing values
df.isnull().sum()

,0
county,0
date,0
fire_label,127078
max_frp,127078
max_brightness,127078
fire_count,127078
temp_max,0
temp_min,0
humidity,0
wind_speed,0


In [ ]:
fire_cols = ['fire_label', 'max_frp', 'max_brightness', 'fire_count']

# Fill the missing fire columns with 0 (no fire detected)
df[fire_cols] = df[fire_cols].fillna(0)

Forward-filling would create fake persistent fires, completely breaking any model we train, especially for next-day prediction.

Gaps are not missing data; they are true negatives (no fire).

In [ ]:
# Check missing values
df.isnull().sum()

,0
county,0
date,0
fire_label,0
max_frp,0
max_brightness,0
fire_count,0
temp_max,0
temp_min,0
humidity,0
wind_speed,0


# Engineer Features

In [ ]:
df = df.sort_values(['county', 'date']).reset_index(drop=True)
df['date'] = pd.to_datetime(df['date'])

df['month'] = df['date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['day_of_year'] = df['date'].dt.dayofyear

# 1 = Sat/Sun
df['weekend_flag'] = (df['date'].dt.weekday >= 5).astype(int)

# 1 = June-November
df['fire_season_flag'] = df['month'].isin([6, 7, 8, 9, 10, 11]).astype(int)

In [ ]:
"""
Drought Index
"""
# Consecutive days with precipitation < 0.1 inch per county
# Resets on rain day (>= 0.1)
def compute_drought_index(group):
    is_dry = group['precipitation'] < 0.1

    # Group consecutive identical values (dry runs)
    change = (is_dry != is_dry.shift(1)).cumsum()

    # Streak length within each run
    streak = is_dry.groupby(change).cumcount() + 1

    # Only keep streak on dry days, else 0
    group['drought_index'] = np.where(is_dry, streak, 0)

    return group

df = df.groupby('county', group_keys=False).apply(compute_drought_index, include_groups=True)

/tmp/ipykernel_5260/867535715.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('county', group_keys=False).apply(compute_drought_index, include_groups=True)


In [ ]:
"""
Rolling Averages (7, 14, 30 days)
"""
# Grouped by county, rolling mean of temp_max and humidity
# min_periods=1 so early days have partial averages (avoids excessive NaNs)
rolling_windows = [7, 14, 30]
for window in rolling_windows:
    df[f'temp_max_{window}d_rolling_mean'] = (
        df.groupby('county')['temp_max']
        .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    )
    df[f'humidity_{window}d_rolling_mean'] = (
        df.groupby('county')['humidity']
        .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    )

In [ ]:
"""
Temperature Anomaly
"""
# temp_max minus 30-day rolling mean (captures heat waves)
df['temperature_anomaly'] = (
    df['temp_max'] - df['temp_max_30d_rolling_mean']
)

"""
VPD (Vapor Pressure Deficit)
"""
# Atmospheric drying power – using temp_max (standard for fire-weather contexts)
df['vpd'] = (
    (1 - df['humidity'] / 100)
    * 0.6108
    * np.exp(17.27 * df['temp_max'] / (df['temp_max'] + 237.3))
)

"""
Interactions
"""
df['wind_speed_drought_interaction'] = df['wind_speed'] * df['drought_index']
df['temp_max_humidity_interaction'] = df['temp_max'] * (100 - df['humidity'])

In [ ]:
df.head()

,county,date,fire_label,max_frp,max_brightness,fire_count,temp_max,temp_min,humidity,wind_speed,...,temp_max_7d_rolling_mean,humidity_7d_rolling_mean,temp_max_14d_rolling_mean,humidity_14d_rolling_mean,temp_max_30d_rolling_mean,humidity_30d_rolling_mean,temperature_anomaly,vpd,wind_speed_drought_interaction,temp_max_humidity_interaction
0,Alameda,2010-01-01,0.0,0.0,0.0,0.0,13.2815,5.7315,84.409645,6.792466,...,13.281500,84.409645,13.281500,84.409645,13.281500,84.409645,0.000000,0.237843,0.000000,207.063300
1,Alameda,2010-01-02,0.0,0.0,0.0,0.0,13.4815,6.1815,95.559820,6.479999,...,13.381500,89.984733,13.381500,89.984733,13.381500,89.984733,0.100000,0.068628,0.000000,59.860287
2,Alameda,2010-01-03,0.0,0.0,0.0,0.0,12.9815,2.8315,84.136185,10.739832,...,13.248167,88.035217,13.248167,88.035217,13.248167,88.035217,-0.266667,0.237316,10.739832,205.936114
3,Alameda,2010-01-04,0.0,0.0,0.0,0.0,13.6315,2.5315,79.060646,7.244860,...,13.344000,85.791574,13.344000,85.791574,13.344000,85.791574,0.287500,0.326816,14.489720,285.434804
4,Alameda,2010-01-05,0.0,0.0,0.0,0.0,13.7315,4.6815,69.012860,7.100310,...,13.421500,82.435831,13.421500,82.435831,13.421500,82.435831,0.310000,0.486796,21.300930,425.499913


In [ ]:
len(df)

233044

In [ ]:
df.isnull().sum()

,0
county,0
date,0
fire_label,0
max_frp,0
max_brightness,0
fire_count,0
temp_max,0
temp_min,0
humidity,0
wind_speed,0


In [ ]:
# Save
df.to_csv('dataset_with_all_features.csv', index=False)